##### 05 - Galaxy Redshift Regression

Train regression models to predict galaxy redshift `z` from photometric
features.

Phase 3 EDA suggested that raw magnitudes have a stronger visible and linear
relationship with redshift than color indices, although both feature groups
will be included in the models.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [4]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
GALAXY_PATH = PROCESSED_DIR / "sdss_galaxy_redshift.csv"

df_galaxy = pd.read_csv(GALAXY_PATH)

df_galaxy.shape

(49402, 18)

In [5]:
features = [
    "dered_u",
    "dered_g",
    "dered_r",
    "dered_i",
    "dered_z",
    "u_g",
    "g_r",
    "r_i",
    "i_z",
]

target = "z"

X = df_galaxy[features].copy()
y = df_galaxy[target].copy()

In [6]:
print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("\nMissing values:")
print(df_galaxy[features + [target]].isnull().sum())

y.describe()

Feature shape: (49402, 9)
Target shape: (49402,)

Missing values:
dered_u    0
dered_g    0
dered_r    0
dered_i    0
dered_z    0
u_g        0
g_r        0
r_i        0
i_z        0
z          0
dtype: int64


count    49402.000000
mean         0.275863
std          0.256418
min          0.000053
25%          0.084763
50%          0.142197
75%          0.455774
max          0.999834
Name: z, dtype: float64

The regression dataset contains only galaxy objects with `0 < z < 1`, and the
selected modeling columns contain no missing values.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
)

X_train.shape, X_test.shape

((39521, 9), (9881, 9))

In [10]:
models = {
    "Dummy": DummyRegressor(strategy="mean"),

    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge()),
    ]),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42,
    ),
}

In [11]:
results = []
predictions = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    trained_models[name] = model
    predictions[name] = y_pred

    results.append({
        "model": name,
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": root_mean_squared_error(y_test, y_pred),
        "r2": r2_score(y_test, y_pred),
    })

results_df = (
    pd.DataFrame(results)
    .sort_values(["mae", "rmse"])
    .reset_index(drop=True)
)

results_df

,model,mae,rmse,r2
0,Random Forest,0.032216,0.066071,0.933607
1,Gradient Boosting,0.036250,0.068174,0.929313
2,Ridge,0.066969,0.102648,0.839749
3,Dummy,0.217133,0.256422,-0.000023


In [12]:
best_model_name = results_df.loc[0, "model"]
best_model = trained_models[best_model_name]
best_pred = predictions[best_model_name]

print("Best model by MAE:", best_model_name)

Best model by MAE: Random Forest
